# LSTM Next-Word Text Prediction

This notebook uses the trained LSTM model from the training notebook to perform next-word prediction.

The trained model, vocabulary, and configuration are loaded from the project files.

The notebook does not perform any model training.

The prediction pipeline is:

User Text
    ↓
Text Preprocessing
    ↓
Word-to-Index Conversion
    ↓
Fixed-Length Input Sequence
    ↓
Trained LSTM Model
    ↓
Next-Word Prediction

In [72]:
import json
import re
import numpy as np
import tensorflow as tf
from pathlib import Path

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "technical_writing_lstm_best.keras"
)

VOCABULARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vocabulary.json"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_config.json"
)

# --------------------------------------------------
# Load trained model
# --------------------------------------------------

model = tf.keras.models.load_model(MODEL_PATH)

# --------------------------------------------------
# Load vocabulary
# --------------------------------------------------

with open(VOCABULARY_PATH, "r", encoding="utf-8") as file:
    vocabulary_data = json.load(file)

word_to_index = vocabulary_data["word_to_index"]

index_to_word = {
    int(index): word
    for index, word in vocabulary_data["index_to_word"].items()
}

# --------------------------------------------------
# Load model configuration
# --------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    model_config = json.load(file)

SEQUENCE_LENGTH = model_config["sequence_length"]
VOCAB_SIZE = model_config["vocabulary_size"]

# --------------------------------------------------
# Special tokens
# --------------------------------------------------

SPECIAL_TOKENS = {
    "<PAD>",
    "<UNK>",
    "<START>",
    "<END>"
}

# --------------------------------------------------
# Text preprocessing
# --------------------------------------------------

def preprocess_text(text):
    text = text.lower()
    tokens = re.findall(r"\b[\w']+\b", text)
    return tokens


def text_to_indices(tokens):
    unk_index = word_to_index.get("<UNK>")

    return [
        word_to_index.get(token, unk_index)
        for token in tokens
    ]


def prepare_input_sequence(text):
    tokens = preprocess_text(text)
    indices = text_to_indices(tokens)

    # Keep only the latest SEQUENCE_LENGTH words
    if len(indices) > SEQUENCE_LENGTH:
        indices = indices[-SEQUENCE_LENGTH:]

    # Left padding
    pad_index = word_to_index.get("<PAD>", 0)

    if len(indices) < SEQUENCE_LENGTH:
        padding_length = SEQUENCE_LENGTH - len(indices)

        indices = (
            [pad_index] * padding_length
            + indices
        )

    return np.array(indices, dtype=np.int32)


print("Model loaded successfully.")
print("Vocabulary loaded successfully.")
print("Configuration loaded successfully.")

print("\nSequence length:", SEQUENCE_LENGTH)
print("Vocabulary size:", VOCAB_SIZE)

Model loaded successfully.
Vocabulary loaded successfully.
Configuration loaded successfully.

Sequence length: 20
Vocabulary size: 10556


## Next-Word Prediction

The trained LSTM predicts the probability of every word in the vocabulary.

For user-facing prediction, special tokens such as `<PAD>`, `<UNK>`, `<START>`, and `<END>` are excluded.

Instead of automatically generating a long sentence, the system returns the most probable normal words as next-word suggestions.

This approach is designed to behave more like predictive typing on a mobile keyboard.

In [73]:
def get_next_word_suggestions(text, number_of_suggestions=3):
    """
    Predict the most likely next words for the given text.
    """

    if not text or not text.strip():
        return []

    # Prepare model input
    input_sequence = prepare_input_sequence(text)
    input_sequence = np.expand_dims(input_sequence, axis=0)

    # Get model probabilities
    probabilities = model.predict(
        input_sequence,
        verbose=0
    )[0]

    # Highest probability first
    sorted_indices = np.argsort(probabilities)[::-1]

    suggestions = []

    for index in sorted_indices:

        index = int(index)

        word = index_to_word.get(
            index,
            "<UNK>"
        )

        # Ignore special tokens
        if word in SPECIAL_TOKENS:
            continue

        suggestions.append({
            "word": word,
            "probability": float(
                probabilities[index]
            )
        })

        if len(suggestions) >= number_of_suggestions:
            break

    return suggestions

## User Input and Next-Word Suggestions

The user can enter text below and receive the predicted next words.

The system displays the top three suggestions along with their model probabilities.

This is the final user-facing prediction interface of the notebook.

The same prediction function will later be reused in the web application.

In [75]:
print("=" * 80)
print("LSTM NEXT-WORD PREDICTOR")
print("=" * 80)

user_text = input("\nEnter your text: ").strip()

if user_text:

    suggestions = get_next_word_suggestions(
        user_text,
        number_of_suggestions=3
    )

    print("\n" + "=" * 80)
    print("PREDICTED NEXT WORDS")
    print("=" * 80)

    print("\nInput:")
    print(user_text)

    print("\nSuggestions:")

    for index, suggestion in enumerate(
        suggestions,
        start=1
    ):
        print(
            f"{index}. "
            f"{suggestion['word']:<20}"
            f"{suggestion['probability'] * 100:.2f}%"
        )

else:
    print("\nNo text was entered.")

LSTM NEXT-WORD PREDICTOR

PREDICTED NEXT WORDS

Input:
letter

Suggestions:
1. the                 5.74%
2. and                 4.52%
3. to                  3.62%
